# Isotools differential splicing analysis (pooled)

## Setup

In [1]:
from isotools import Transcriptome
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
path='/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output'
isoseq=Transcriptome.load(f'{path}/A_najas_isotools_filtered_pooled.pkl')

In [3]:
import numpy as np
import sys
import io
from contextlib import redirect_stderr
from statsmodels.stats.multitest import multipletests

# Define parameters
outpath = "/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS"  
types_of_interest = ['ES', 'ME', '5AS', '3AS', 'IR']  # Ignore TSS and PAS

## I4 vs I5

In [4]:
group1_name = 'I4'
group2_name = 'I5'

group_dict = {
    group1_name: ['I4'],           # I4 group contains just I4 sample
    group2_name: ['I5F', 'I5M']    # I5 group contains I5F and I5M samples
}

pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Group 1 ({group1_name}): {group_dict[group1_name]}")
print(f"Group 2 ({group2_name}): {group_dict[group2_name]}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing I4 vs I5
Group 1 (I4): ['I4']
Group 2 (I5): ['I5F', 'I5M']
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1975 problematic genes out of 15472
Saved problematic genes to 'I4_I5_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 4964 events across 2290 genes
Skipped 1975 problematic genes

RESULTS:
  3051 differential splice sites in 1473 genes

Significant events by type:
  ES: 385
  ME: 113
  5AS: 566
  3AS: 447
  IR: 1540

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/I4_I5/I4_I5_differential_events.csv'
Updated problematic genes saved to 'I4_I5_problematic_genes_updated.txt'


## I5 vs AD

In [5]:
group1_name = 'I5'
group2_name = 'AD'

group_dict = {
    group1_name: ['I5F', "I5M"],   # I5 group contains I5F and I5M samples
    group2_name: ['Fad', 'Mad']    # AD group contains Fad and Mad samples
}

pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Group 1 ({group1_name}): {group_dict[group1_name]}")
print(f"Group 2 ({group2_name}): {group_dict[group2_name]}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing I5 vs AD
Group 1 (I5): ['I5F', 'I5M']
Group 2 (AD): ['Fad', 'Mad']
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1881 problematic genes out of 15472
Saved problematic genes to 'I5_AD_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 4498 events across 2156 genes
Skipped 1881 problematic genes

RESULTS:
  2728 differential splice sites in 1316 genes

Significant events by type:
  ES: 342
  ME: 91
  5AS: 521
  3AS: 381
  IR: 1393

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/I5_AD/I5_AD_differential_events.csv'
Updated problematic genes saved to 'I5_AD_problematic_genes_updated.txt'
